# 作业 3：卷积神经网络相关基础

姓名：付航  
学号：20234080304  


In [1]:
# 环境检查单元格
# 作用：确保 torch 和 torchvision 安装在“当前 Jupyter Notebook 使用的 Python 内核”中。
# 注意：如果你已经安装过，这个单元格只会打印版本，不会重复安装。

import sys
import subprocess
import importlib.util

print("当前 Notebook 使用的 Python 解释器：")
print(sys.executable)

def package_exists(pkg_name):
    return importlib.util.find_spec(pkg_name) is not None

need_install = []

if not package_exists("torch"):
    need_install.append("torch")

if not package_exists("torchvision"):
    need_install.append("torchvision")

if need_install:
    print("缺少以下包：", need_install)
    print("正在安装到当前 Notebook 内核对应的 Python 环境中……")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        *need_install,
        "-i", "https://pypi.tuna.tsinghua.edu.cn/simple"
    ])
else:
    print("torch 和 torchvision 均已安装。")

import torch
import torchvision

print("torch 版本：", torch.__version__)
print("torchvision 版本：", torchvision.__version__)
print("环境检查完成。")

当前 Notebook 使用的 Python 解释器：
f:\anconda\python.exe
torch 和 torchvision 均已安装。
torch 版本： 2.12.0+cpu
torchvision 版本： 0.27.0+cpu
环境检查完成。


## 2.1 卷积层理论计算题

输入图像大小为：$3 \times 32 \times 32$  
卷积核数量为 16，每个卷积核大小为：$3 \times 5 \times 5$  
Padding = 2，Stride = 2。

卷积输出高宽计算公式：

$$
H_{out}=\left\lfloor \frac{H + 2P - K}{S} \right\rfloor + 1
$$

代入：

$$
H_{out}=W_{out}=\left\lfloor \frac{32 + 2\times 2 - 5}{2} \right\rfloor + 1
=\left\lfloor \frac{31}{2} \right\rfloor + 1
=15+1=16
$$

因此输出特征图尺寸为：

$$
16 \times 16 \times 16
$$

单个输出通道的一个像素值，需要与输入的一个 $3 \times 5 \times 5$ 感受野进行乘法运算，因此乘法次数为：

$$
3 \times 5 \times 5 = 75
$$

答案：
1. 输出尺寸：$16 \times 16 \times 16$
2. 单个输出通道一个像素值需要 75 次乘法操作。

In [2]:
import numpy as np

def _to_pair(value):
    """将 int 或 tuple/list 转换为二元组。"""
    if isinstance(value, int):
        return value, value
    if isinstance(value, (tuple, list)) and len(value) == 2:
        return int(value[0]), int(value[1])
    raise ValueError("参数必须是 int 或长度为 2 的 tuple/list")

def max_pool2d_numpy(x, kernel_size, stride=1, padding=0):
    """
    使用 NumPy 手动实现二维最大池化前向传播。

    参数：
    x: 输入数组，形状为 (N, C, H, W)
    kernel_size: 池化窗口大小，可以是 int 或 (kh, kw)
    stride: 步幅，可以是 int 或 (sh, sw)
    padding: 填充，可以是 int 或 (ph, pw)

    返回：
    输出数组，形状为 (N, C, out_h, out_w)
    """
    x = np.asarray(x)

    if x.ndim != 4:
        raise ValueError("输入 x 的形状必须为 (N, C, H, W)")

    kh, kw = _to_pair(kernel_size)
    sh, sw = _to_pair(stride)
    ph, pw = _to_pair(padding)

    # 修正点：
    # 如果 x 是整数类型，不能直接 padding -np.inf，否则会出现 OverflowError。
    # 因此这里统一转为 float32，保证 -np.inf 可以正常填充。
    x = x.astype(np.float32)

    N, C, H, W = x.shape

    x_padded = np.pad(
        x,
        pad_width=((0, 0), (0, 0), (ph, ph), (pw, pw)),
        mode="constant",
        constant_values=-np.inf
    )

    H_p, W_p = x_padded.shape[2], x_padded.shape[3]
    out_h = (H_p - kh) // sh + 1
    out_w = (W_p - kw) // sw + 1

    if out_h <= 0 or out_w <= 0:
        raise ValueError("池化窗口过大或参数设置不合理，导致输出尺寸小于等于 0")

    out = np.empty((N, C, out_h, out_w), dtype=x.dtype)

    for n in range(N):
        for c in range(C):
            for i in range(out_h):
                for j in range(out_w):
                    h_start = i * sh
                    h_end = h_start + kh
                    w_start = j * sw
                    w_end = w_start + kw
                    window = x_padded[n, c, h_start:h_end, w_start:w_end]
                    out[n, c, i, j] = np.max(window)

    return out

# 测试
x = np.arange(1, 17).reshape(1, 1, 4, 4)
print("输入：")
print(x)

print("\n最大池化结果 kernel_size=2, stride=2, padding=0：")
print(max_pool2d_numpy(x, kernel_size=2, stride=2, padding=0))

print("\n最大池化结果 kernel_size=2, stride=1, padding=1：")
print(max_pool2d_numpy(x, kernel_size=2, stride=1, padding=1))

输入：
[[[[ 1  2  3  4]
   [ 5  6  7  8]
   [ 9 10 11 12]
   [13 14 15 16]]]]

最大池化结果 kernel_size=2, stride=2, padding=0：
[[[[ 6.  8.]
   [14. 16.]]]]

最大池化结果 kernel_size=2, stride=1, padding=1：
[[[[ 1.  2.  3.  4.  4.]
   [ 5.  6.  7.  8.  8.]
   [ 9. 10. 11. 12. 12.]
   [13. 14. 15. 16. 16.]
   [13. 14. 15. 16. 16.]]]]


## 3.1 VGG 理论计算题

假设输入和输出通道数均为 $C$。

### 1. 一个 $5 \times 5$ 卷积层参数量

不带偏置时：

$$
5 \times 5 \times C \times C = 25C^2
$$

### 2. 两个串联的 $3 \times 3$ 卷积层参数量

每个 $3 \times 3$ 卷积层参数量为：

$$
3 \times 3 \times C \times C = 9C^2
$$

两个串联：

$$
2 \times 9C^2 = 18C^2
$$

结论：两个 $3 \times 3$ 卷积层参数量更少，同时可以引入更多非线性表达能力。

In [3]:
import torch
from torch import nn

def nin_block(in_channels, out_channels, kernel_size, stride, padding):
    """
    定义 NiN Block：
    普通卷积层 + ReLU
    1x1 卷积层 + ReLU
    1x1 卷积层 + ReLU
    """
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU()
    )

# 测试
block = nin_block(3, 16, kernel_size=5, stride=1, padding=2)
x = torch.randn(2, 3, 32, 32)
y = block(x)

print(block)
print("输入形状：", x.shape)
print("输出形状：", y.shape)

Sequential(
  (0): Conv2d(3, 16, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (1): ReLU()
  (2): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1))
  (3): ReLU()
  (4): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1))
  (5): ReLU()
)
输入形状： torch.Size([2, 3, 32, 32])
输出形状： torch.Size([2, 16, 32, 32])


## 4.1 Batch Normalization 理论计算题

给定：

$$
x_1=2, x_2=4, x_3=6, x_4=8
$$

均值：

$$
\mu = \frac{2+4+6+8}{4}=5
$$

方差：

$$
\sigma^2 = \frac{(2-5)^2+(4-5)^2+(6-5)^2+(8-5)^2}{4}
=\frac{9+1+1+9}{4}=5
$$

标准差：

$$
\sqrt{\sigma^2+\epsilon}=\sqrt{5}
$$

归一化结果：

$$
\hat{x_i}=\frac{x_i-5}{\sqrt{5}}
$$

最终输出：

$$
y_i=\gamma \hat{x_i}+\beta
$$

其中 $\gamma=2, \beta=1$，所以：

$$
y_i = 2\times \frac{x_i-5}{\sqrt{5}} + 1
$$

四个输出分别为：

$$
y_1 = 1 - \frac{6}{\sqrt{5}} \approx -1.6833
$$

$$
y_2 = 1 - \frac{2}{\sqrt{5}} \approx 0.1056
$$

$$
y_3 = 1 + \frac{2}{\sqrt{5}} \approx 1.8944
$$

$$
y_4 = 1 + \frac{6}{\sqrt{5}} \approx 3.6833
$$

In [4]:
class Residual(nn.Module):
    """
    ResNet 残差块。

    包含：
    - 两个 3x3 卷积层
    - 每个卷积层后接 BatchNorm
    - 如果 use_1x1conv=True，则使用 1x1 卷积调整输入形状
    """
    def __init__(self, input_channels, num_channels, use_1x1conv=False, strides=1):
        super().__init__()
        self.conv1 = nn.Conv2d(
            input_channels, num_channels,
            kernel_size=3, padding=1, stride=strides
        )
        self.bn1 = nn.BatchNorm2d(num_channels)
        self.conv2 = nn.Conv2d(
            num_channels, num_channels,
            kernel_size=3, padding=1
        )
        self.bn2 = nn.BatchNorm2d(num_channels)

        if use_1x1conv:
            self.conv3 = nn.Conv2d(
                input_channels, num_channels,
                kernel_size=1, stride=strides
            )
        else:
            self.conv3 = None

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        y = self.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))

        if self.conv3:
            x = self.conv3(x)

        y = y + x
        return self.relu(y)

# 测试 1：输入输出形状相同
block1 = Residual(3, 3)
x1 = torch.randn(2, 3, 32, 32)
y1 = block1(x1)
print("测试 1 输入形状：", x1.shape)
print("测试 1 输出形状：", y1.shape)

# 测试 2：使用 1x1 卷积改变通道数和空间尺寸
block2 = Residual(3, 16, use_1x1conv=True, strides=2)
x2 = torch.randn(2, 3, 32, 32)
y2 = block2(x2)
print("测试 2 输入形状：", x2.shape)
print("测试 2 输出形状：", y2.shape)

测试 1 输入形状： torch.Size([2, 3, 32, 32])
测试 1 输出形状： torch.Size([2, 3, 32, 32])
测试 2 输入形状： torch.Size([2, 3, 32, 32])
测试 2 输出形状： torch.Size([2, 16, 16, 16])


## 5.1 微调理论题

### 1. 为什么底层特征提取层学习率较小，而顶层输出层学习率较大？

在预训练模型中，底层卷积层通常学习到的是通用视觉特征，例如边缘、纹理、颜色和简单形状。这些特征在不同图像任务之间具有较强的迁移能力，因此不需要大幅度更新。如果对这些层使用过大的学习率，可能会破坏预训练模型已经学到的有效特征。

而最终输出层通常需要根据新的目标数据集重新初始化。例如 ImageNet 是 1000 类分类，而新任务可能只有少数类别。新初始化的输出层没有经过充分训练，因此需要较大的学习率，使其更快适应新任务。

### 2. 目标数据集非常小且与源数据集非常相似时，应采取什么微调策略？

如果目标数据集很小，并且与源数据集非常相似，应尽量减少需要训练的参数，以防止过拟合。常见策略是：

1. 冻结大部分底层特征提取层；
2. 只训练最后的分类层；
3. 对少量高层特征层使用较小学习率进行微调；
4. 使用数据增广、权重衰减、Dropout、早停等方法提升泛化能力。

In [5]:
from torchvision import transforms
from PIL import Image
import numpy as np

# 1. 按题目要求创建图像增广 Pipeline
augmentation_pipeline = transforms.Compose([
    transforms.RandomResizedCrop(
        size=224,
        scale=(0.08, 1.0)
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(
        brightness=0.5,
        contrast=0.5,
        saturation=0.5
    ),
    transforms.ToTensor()
])

print("图像增广 Pipeline：")
print(augmentation_pipeline)

# 2. 为了满足“编程题需要打印相应输出”的要求，
#    这里不依赖外部图片，而是随机生成一张 RGB 测试图片。
#    这样老师运行 Notebook 时不需要额外准备图片文件。
random_image_array = np.random.randint(
    low=0,
    high=256,
    size=(300, 300, 3),
    dtype=np.uint8
)

test_image = Image.fromarray(random_image_array)

# 3. 对测试图片应用图像增广
augmented_tensor = augmentation_pipeline(test_image)

print("\n随机测试图片原始尺寸：")
print(test_image.size)

print("\n增广后 Tensor 形状：")
print(augmented_tensor.shape)

print("\n增广后 Tensor 数值范围：")
print("min =", augmented_tensor.min().item())
print("max =", augmented_tensor.max().item())

图像增广 Pipeline：
Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
    RandomHorizontalFlip(p=0.5)
    ColorJitter(brightness=(0.5, 1.5), contrast=(0.5, 1.5), saturation=(0.5, 1.5), hue=None)
    ToTensor()
)

随机测试图片原始尺寸：
(300, 300)

增广后 Tensor 形状：
torch.Size([3, 224, 224])

增广后 Tensor 数值范围：
min = 0.2666666805744171
max = 1.0


## 6.1 IoU 理论计算题

真实框：

$$
A=[10,10,50,50]
$$

预测框：

$$
B=[30,30,70,70]
$$

两个框的交集左上角坐标为：

$$
(\max(10,30), \max(10,30))=(30,30)
$$

交集右下角坐标为：

$$
(\min(50,70), \min(50,70))=(50,50)
$$

交集宽高为：

$$
50-30=20
$$

所以交集面积：

$$
20\times 20=400
$$

A 的面积：

$$
(50-10)\times(50-10)=40\times40=1600
$$

B 的面积：

$$
(70-30)\times(70-30)=40\times40=1600
$$

并集面积：

$$
1600+1600-400=2800
$$

因此：

$$
IoU=\frac{400}{2800}=\frac{1}{7}\approx0.142857
$$

In [6]:
import torch
import torch.nn.functional as F

def label_smoothing_cross_entropy(logits, target, epsilon=0.1, reduction="mean"):
    """
    计算标签平滑后的交叉熵损失。

    参数：
    logits: 模型输出，形状为 (batch_size, num_classes)
    target: 真实标签，形状为 (batch_size,)
    epsilon: 标签平滑因子
    reduction: "mean", "sum", "none"
    """
    if logits.ndim != 2:
        raise ValueError("logits 的形状必须为 (batch_size, num_classes)")

    num_classes = logits.size(1)

    if num_classes <= 1:
        raise ValueError("分类类别数必须大于 1")

    log_probs = F.log_softmax(logits, dim=1)

    with torch.no_grad():
        true_dist = torch.full_like(log_probs, epsilon / (num_classes - 1))
        true_dist.scatter_(1, target.unsqueeze(1), 1.0 - epsilon)

    loss = -torch.sum(true_dist * log_probs, dim=1)

    if reduction == "mean":
        return loss.mean()
    elif reduction == "sum":
        return loss.sum()
    elif reduction == "none":
        return loss
    else:
        raise ValueError("reduction 必须是 'mean'、'sum' 或 'none'")

# 测试
logits = torch.tensor([
    [2.0, 0.5, 0.1],
    [0.2, 1.5, 0.3]
])
target = torch.tensor([0, 1])

loss = label_smoothing_cross_entropy(logits, target, epsilon=0.1)
print("标签平滑交叉熵损失：", loss.item())

# 显示平滑后的标签分布
num_classes = logits.size(1)
epsilon = 0.1
smooth_labels = torch.full_like(logits, epsilon / (num_classes - 1))
smooth_labels.scatter_(1, target.unsqueeze(1), 1.0 - epsilon)
print("平滑后的标签分布：")
print(smooth_labels)

标签平滑交叉熵损失： 0.5326123833656311
平滑后的标签分布：
tensor([[0.9000, 0.0500, 0.0500],
        [0.0500, 0.9000, 0.0500]])
